# OT-2 Simulation Tests

Simulation mode runs a compiled protocol through **Opentrons' own simulator** rather than
faking the robot's HTTP replies. That matters: `opentrons.simulate` executes the protocol
against the same analysis engine the instrument uses, so it catches unknown labware, a
volume beyond a pipette's capacity, or a Flex protocol with no trash defined. Mocked HTTP
responses report success for all of those.

**Prerequisite:** `pip install 'ot2_module[simulation]'`. The simulator is an optional
dependency because it adds about 54 MB that a production container never imports.

Part 1 needs nothing but the package. Part 2 needs a reachable MADSci resource manager.


In [1]:
import os, subprocess, sys, time, urllib.request
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == 'tests' else Path.cwd()
SRC = REPO / 'src'
sys.path.insert(0, str(SRC))

NODE_PORT = 2210
NODE_URL = f'http://127.0.0.1:{NODE_PORT}'
RESOURCE_SERVER_URL = os.environ.get('RESOURCE_SERVER_URL', 'http://127.0.0.1:8003')
print('repo           ', REPO)
print('resource server', RESOURCE_SERVER_URL)


repo            /Users/dozgulbas/workspace/ot2_module
resource server http://127.0.0.1:9004


In [2]:
try:
    import opentrons
    print('opentrons', opentrons.__version__)
except ImportError:
    raise SystemExit("opentrons is missing. pip install 'ot2_module[simulation]'")


opentrons 8.8.2


## Part 1. The driver, no robot and no managers

`OT2_Driver.__init__` performs no I/O. A real node calls `connect()` after constructing;
a simulation node never does, so it cannot reach an instrument whatever else happens.


In [3]:
from ot2_interface.ot2_driver_http import OT2_Driver
from ot2_interface.config import OT2_Config

# 192.0.2.1 is TEST-NET-1, reserved and guaranteed unroutable. If construction
# touched the network this would hang or raise.
driver = OT2_Driver(OT2_Config(ip='192.0.2.1'), simulation=True)
print('constructed with no connection, simulation =', driver.simulation)


constructed with no connection, simulation = True


In [4]:
try:
    driver.connect()
    print('FAIL: connect() did not refuse')
except RuntimeError as e:
    print('PASS: connect() refused ->', e)


PASS: connect() refused -> connect() called on an OT2_Driver built in simulation mode.


### Test protocols

One valid, two deliberately broken. `apiLevel 2.12` is what protopiler emits.


In [5]:
import tempfile
TMP = Path(tempfile.mkdtemp())

VALID = TMP / 'valid.py'
VALID.write_text('''
from opentrons import protocol_api
metadata = {"protocolName": "valid", "apiLevel": "2.12"}
def run(protocol: protocol_api.ProtocolContext):
    tips = protocol.load_labware("opentrons_96_tiprack_300ul", 1)
    plate = protocol.load_labware("corning_96_wellplate_360ul_flat", 2)
    p = protocol.load_instrument("p300_single_gen2", "right", tip_racks=[tips])
    p.transfer(100, plate["A1"], plate["B1"])
'''.lstrip())

BAD_LABWARE = TMP / 'bad_labware.py'
BAD_LABWARE.write_text('''
from opentrons import protocol_api
metadata = {"protocolName": "bad labware", "apiLevel": "2.12"}
def run(protocol: protocol_api.ProtocolContext):
    protocol.load_labware("this_labware_does_not_exist_96", 1)
'''.lstrip())

BAD_VOLUME = TMP / 'bad_volume.py'
BAD_VOLUME.write_text('''
from opentrons import protocol_api
metadata = {"protocolName": "overdraw", "apiLevel": "2.12"}
def run(protocol: protocol_api.ProtocolContext):
    tips = protocol.load_labware("opentrons_96_tiprack_300ul", 1)
    plate = protocol.load_labware("corning_96_wellplate_360ul_flat", 2)
    p = protocol.load_instrument("p300_single_gen2", "right", tip_racks=[tips])
    p.pick_up_tip()
    p.aspirate(5000, plate["A1"])   # a p300 holds 300 uL
'''.lstrip())
print('wrote 3 protocols to', TMP)


wrote 3 protocols to /var/folders/8x/vpjpyrt54gs63wc6mn6k6tnh0000gp/T/tmpuzycy1il


In [6]:
results = []

# Only an error raised by the simulator counts as a rejection. Anything else, a
# NameError from a cell that did not run, a missing dependency, is a broken test
# and must not be scored as a correct refusal. An earlier version of this harness
# caught bare Exception and reported two false passes when the driver cell failed.
SIMULATOR_ERRORS = ('ExceptionInProtocolError', 'ProtocolEngineExecuteError',
                    'MalformedProtocolError', 'ApiDeprecationError', 'RuntimeError')

def check(name, path, expect):
    """expect is 'accepted' or 'rejected'. Raises if the test itself is broken."""
    try:
        _pid, run_id = driver.transfer(str(path))
        n = driver.execute(run_id)['data']['commandCount']
        got, detail = 'accepted', f'{n} commands'
    except Exception as e:
        if type(e).__name__ not in SIMULATOR_ERRORS:
            raise
        got = 'rejected'
        detail = f'{type(e).__name__}: {str(e)[:110]}'
    ok = got == expect
    results.append((name, ok))
    print(f"[{'PASS' if ok else 'FAIL'}] {name}: {got}")
    print(f'       {detail}')
    print()


### The point of the whole exercise

A mocked HTTP layer returns `200 OK` for all three of these. The simulator does not.


In [7]:
check('valid protocol', VALID, 'accepted')


/Users/dozgulbas/.opentrons/robot_settings.json not found. Loading defaults


Deck calibration not found.


/Users/dozgulbas/.opentrons/deck_calibration.json not found. Loading defaults


[PASS] valid protocol: accepted
       5 commands



In [8]:
check('unknown labware', BAD_LABWARE, 'rejected')


/Users/dozgulbas/.opentrons/robot_settings.json not found. Loading defaults


Deck calibration not found.


/Users/dozgulbas/.opentrons/deck_calibration.json not found. Loading defaults


[PASS] unknown labware: rejected
       ExceptionInProtocolError: FileNotFoundError [line 4]: Unable to find a labware
        definition for "this_labware_does_not_exist_96",




In [9]:
check('aspirate 5000uL with a p300', BAD_VOLUME, 'rejected')


/Users/dozgulbas/.opentrons/robot_settings.json not found. Loading defaults


Deck calibration not found.


/Users/dozgulbas/.opentrons/deck_calibration.json not found. Loading defaults


[PASS] aspirate 5000uL with a p300: rejected
       ExceptionInProtocolError: AssertionError [line 8]: Cannot aspirate more than pipette max volume



### A Flex protocol

opentrons 8.8.2 handles the Flex too, up to apiLevel 2.27, which is why
`otflex_rest_node` is covered by the same mechanism.


`opentrons.simulate` calls `asyncio.run()` on the Flex path, which raises inside a
notebook because the kernel already has a running event loop. That is Jupyter's
constraint rather than the product's, so this check runs in a subprocess. The OT-2
path above does not hit it.


In [10]:
FLEX = TMP / 'flex.py'
FLEX.write_text('''
from opentrons import protocol_api
metadata = {"protocolName": "flex"}
requirements = {"robotType": "Flex", "apiLevel": "2.18"}
def run(protocol: protocol_api.ProtocolContext):
    protocol.load_trash_bin("A3")
    tips = protocol.load_labware("opentrons_flex_96_tiprack_200ul", "A1")
    plate = protocol.load_labware("corning_96_wellplate_360ul_flat", "B1")
    p = protocol.load_instrument("flex_1channel_1000", "left", tip_racks=[tips])
    p.transfer(100, plate["A1"], plate["B1"])
'''.lstrip())

# Subprocess, to escape the notebook's event loop.
probe = TMP / 'flex_probe.py'
probe.write_text(
    'import sys\n'
    f'sys.path.insert(0, {str(SRC)!r})\n'
    'from ot2_interface.ot2_driver_http import OT2_Driver\n'
    'from ot2_interface.config import OT2_Config\n'
    'd = OT2_Driver(OT2_Config(ip="192.0.2.1"), simulation=True)\n'
    f'_pid, rid = d.transfer({str(FLEX)!r})\n'
    'print(d.execute(rid)["data"]["commandCount"])\n'
)
r = subprocess.run([sys.executable, str(probe)], capture_output=True,
                   text=True, timeout=300)
ok = r.returncode == 0
results.append(('Flex protocol', ok))
if ok:
    print(f'[PASS] Flex protocol: accepted, {r.stdout.strip().splitlines()[-1]} commands')
else:
    print('[FAIL] Flex protocol: rejected')
    print('      ', (r.stderr.strip().splitlines() or [''])[-1][:120])


[PASS] Flex protocol: accepted, 5 commands


## Part 2. The node

Needs a reachable resource manager. Point it at a **sandbox**, not a live lab:
the node writes resources exactly as a real one does, which is deliberate, because
a multi-step workflow can only be validated if step 2 sees what step 1 did.


In [11]:
def manager_reachable(url):
    try:
        urllib.request.urlopen(url.rstrip('/') + '/health', timeout=3)
        return True
    except Exception:
        return False

HAVE_MANAGER = manager_reachable(RESOURCE_SERVER_URL)
print('resource manager reachable:', HAVE_MANAGER)
if not HAVE_MANAGER:
    print('Part 2 will be skipped. Start a sandbox resource manager and set',
          'RESOURCE_SERVER_URL to run it.')


resource manager reachable: True


You can point this at a node that is **already running**, which is how to exercise
the Docker image rather than a native process:

```bash
OT2_NODE_URL=http://127.0.0.1:2310 jupyter nbconvert --execute ...
```

Left unset, the notebook starts its own node. Note that a native start currently
fails on a MADSci issue unrelated to this module: a pydantic serialization error in
node_types.py reached from abstract_node_module.start_node, reproducible with no
opentrons and no simulation flag. Container images are unaffected because they
froze an older pydantic, so the Docker route is the reliable one today.


In [12]:
EXTERNAL_NODE = os.environ.get('OT2_NODE_URL')
if EXTERNAL_NODE:
    NODE_URL = EXTERNAL_NODE
    print('testing an already-running node at', NODE_URL)
else:
    print('will start a node locally on', NODE_URL)


testing an already-running node at http://127.0.0.1:2310


In [13]:
proc = None
if HAVE_MANAGER and not EXTERNAL_NODE:
    env = {**os.environ,
           'PYTHONPATH': str(SRC),
           'RESOURCE_SERVER_URL': RESOURCE_SERVER_URL,
           'NODE_SIMULATION': 'true',
           'NODE_NAME': 'ot2_sim_notebook',
           # NODE_URL, not NODE_NODE_URL. A field whose own name starts with
           # node_ is not doubled by the NODE_ prefix, and getting it wrong
           # fails silently and leaves the node on its default port.
           'NODE_URL': f'http://0.0.0.0:{NODE_PORT}'}
    log = open('/tmp/ot2_sim_notebook.log', 'w')
    proc = subprocess.Popen([sys.executable, '-m', 'ot2_rest_node'],
                            env=env, stdout=log, stderr=subprocess.STDOUT)
    print('started pid', proc.pid)


In [14]:
node_up = False
if HAVE_MANAGER:
    deadline = time.time() + 90
    up = False
    while time.time() < deadline:
        try:
            urllib.request.urlopen(NODE_URL + '/status', timeout=3)
            up = True; break
        except Exception:
            time.sleep(1)
    node_up = up
    print('node up:', up)
    if not up and not EXTERNAL_NODE:
        print('see /tmp/ot2_sim_notebook.log')


node up: True


In [15]:
if node_up:
    from madsci.client.node.rest_node_client import RestNodeClient
    client = RestNodeClient(url=NODE_URL)
    print(client.get_status())
    info = client.get_info()
    print('actions:', sorted(info.actions.keys()))
    results.append(('node reports ready', client.get_status().ready))
else:
    print('skipped: no node to talk to')


busy=False running_actions=set() paused=False locked=False stopped=False errored=False disconnected=False errors=[] initializing=False waiting_for_config=set() config_values={} ready=True description='Node is ready'
actions: ['run_protocol']


## Part 3. Results and teardown

Teardown runs before the assertion on purpose. Asserting first means a failing run
leaves its node behind, which is exactly the run you least want leaking.


In [16]:
if proc is not None:
    proc.terminate()
    proc.wait(timeout=20)
    print('node stopped')
import shutil; shutil.rmtree(TMP, ignore_errors=True)
print('temp protocols removed')


temp protocols removed


In [17]:
passed = sum(1 for _, ok in results if ok)
print(f'{passed}/{len(results)} passed')
for name, ok in results:
    if not ok:
        print('  FAILED:', name)


5/5 passed


In [18]:
assert passed == len(results), f'{len(results) - passed} test(s) failed'
print('all checks passed')


all checks passed
